# Representative Electric Full Correlation EDA

대표 전기 계량기 11개에 대해 raw DB 기준 전체 컬럼 correlation matrix를 확인하는 노트북입니다.

- 6년 전체
- 연도별
- 계절별

기본 스크립트 `scripts/eda_raw_correlation_representative_electric.py`의 helper를 재사용합니다.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'raw_eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.eda_raw_correlation_representative_electric import (
    REPRESENTATIVE_METERS,
    SEASON_ORDER,
    add_period_columns,
    build_corr_matrix,
    build_engine,
    fetch_meter_raw_df,
    fetch_weather_df,
    select_usable_columns,
)

pio.renderers.default = 'notebook_connected'

In [2]:
REPRESENTATIVE_METERS

['H1.Z10',
 'H1.Z16',
 'H1.Z13',
 'H2.Z64',
 'H4.Z50',
 'H2.Z68',
 'V.Z84',
 'H1.Z20',
 'H2.T.Z33',
 'H2.Z35',
 'H2.ZE64']

In [3]:
engine = build_engine()
weather_df, weather_columns = fetch_weather_df(engine)
weather_columns

['Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']

In [4]:
def load_meter_df(meter_urn: str):
    raw_df, measurements = fetch_meter_raw_df(engine, meter_urn)
    merged_df = raw_df.merge(weather_df, on='ts', how='left')
    merged_df = add_period_columns(merged_df)
    candidate_columns = measurements + weather_columns
    usable_columns = select_usable_columns(merged_df, candidate_columns)
    return merged_df, measurements, usable_columns


def plot_full_corr(df: pd.DataFrame, columns: list[str], title: str, html_name: str | None = None):
    corr_df = build_corr_matrix(df, columns)
    fig = go.Figure(
        data=go.Heatmap(
            z=corr_df.values,
            x=columns,
            y=columns,
            zmin=-1,
            zmax=1,
            colorscale='RdBu',
            reversescale=True,
            colorbar=dict(title='corr'),
            text=corr_df.round(2).values,
            texttemplate='%{text}',
            textfont=dict(size=9),
            hovertemplate='x=%{x}<br>y=%{y}<br>corr=%{z:.4f}<extra></extra>',
        )
    )
    fig.update_layout(
        title=title,
        width=max(900, len(columns) * 36),
        height=max(800, len(columns) * 32),
        xaxis=dict(tickangle=90),
        yaxis=dict(autorange='reversed'),
    )
    if html_name is not None:
        fig.write_html(OUTPUT_DIR / html_name, include_plotlyjs='cdn')
    fig.show()
    return corr_df


def show_meter_6year(meter_urn: str):
    merged_df, measurements, usable_columns = load_meter_df(meter_urn)
    print('meter_urn =', meter_urn)
    print('measurement columns =', measurements)
    print('usable column count =', len(usable_columns))
    print('usable columns =', usable_columns)
    return plot_full_corr(merged_df, usable_columns, f'{meter_urn} 6-Year Full Correlation Matrix', f'{meter_urn}_6year_full_corr.html')


def show_meter_year(meter_urn: str, year: int):
    merged_df, _, usable_columns = load_meter_df(meter_urn)
    year_df = merged_df.loc[merged_df['year'] == year].copy()
    year_columns = select_usable_columns(year_df, usable_columns)
    print('meter_urn =', meter_urn, '| year =', year)
    print('rows =', len(year_df))
    print('usable columns =', year_columns)
    return plot_full_corr(year_df, year_columns, f'{meter_urn} {year} Full Correlation Matrix', f'{meter_urn}_{year}_full_corr.html')


def show_meter_season(meter_urn: str, season: str):
    merged_df, _, usable_columns = load_meter_df(meter_urn)
    season_df = merged_df.loc[merged_df['season'] == season].copy()
    season_columns = select_usable_columns(season_df, usable_columns)
    print('meter_urn =', meter_urn, '| season =', season)
    print('rows =', len(season_df))
    print('usable columns =', season_columns)
    return plot_full_corr(season_df, season_columns, f'{meter_urn} {season} Full Correlation Matrix', f'{meter_urn}_{season}_full_corr.html')

## Example: 6-Year Full Matrix

In [5]:
corr_6year = show_meter_6year('H1.Z16')
corr_6year.round(3)

meter_urn = H1.Z16
measurement columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']
usable column count = 30
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


,I1,I2,I3,P,P1,P2,P3,PF,PF1,PF2,...,Ah,Dc,Dp,H,Igc,Igm,Sc,Ta,Ua,rho
I1,1.000,0.999,0.999,0.998,0.998,0.998,0.998,-0.955,-0.927,-0.762,...,0.039,0.029,0.041,0.053,0.056,0.057,0.064,0.058,-0.042,-0.054
I2,0.999,1.000,0.999,0.998,0.997,0.998,0.997,-0.955,-0.927,-0.762,...,0.038,0.029,0.040,0.053,0.056,0.057,0.064,0.058,-0.041,-0.054
I3,0.999,0.999,1.000,0.996,0.995,0.995,0.998,-0.957,-0.927,-0.763,...,0.039,0.029,0.041,0.054,0.056,0.057,0.065,0.059,-0.042,-0.055
P,0.998,0.998,0.996,1.000,1.000,1.000,0.998,-0.937,-0.908,-0.730,...,0.039,0.029,0.042,0.055,0.059,0.060,0.063,0.062,-0.045,-0.057
P1,0.998,0.997,0.995,1.000,1.000,1.000,0.997,-0.936,-0.907,-0.729,...,0.039,0.028,0.042,0.055,0.059,0.060,0.063,0.062,-0.045,-0.057
P2,0.998,0.998,0.995,1.000,1.000,1.000,0.997,-0.937,-0.908,-0.730,...,0.039,0.029,0.041,0.055,0.059,0.060,0.063,0.061,-0.045,-0.057
P3,0.998,0.997,0.998,0.998,0.997,0.997,1.000,-0.941,-0.907,-0.731,...,0.041,0.028,0.043,0.057,0.060,0.061,0.063,0.063,-0.045,-0.058
PF,-0.955,-0.955,-0.957,-0.937,-0.936,-0.937,-0.941,1.000,0.992,0.911,...,-0.031,-0.031,-0.031,-0.036,-0.035,-0.036,-0.066,-0.036,0.021,0.035
PF1,-0.927,-0.927,-0.927,-0.908,-0.907,-0.908,-0.907,0.992,1.000,0.936,...,-0.020,-0.031,-0.019,-0.023,-0.032,-0.033,-0.069,-0.022,0.015,0.023
PF2,-0.762,-0.762,-0.763,-0.730,-0.729,-0.730,-0.731,0.911,0.936,1.000,...,-0.028,-0.029,-0.027,-0.018,0.021,0.020,-0.056,-0.010,-0.014,0.012


## Example: Yearly Full Matrix

In [6]:
corr_2023 = show_meter_year('H1.Z16', 2023)
corr_2023.round(3)

meter_urn = H1.Z16 | year = 2023
rows = 8760
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


,I1,I2,I3,P,P1,P2,P3,PF,PF1,PF2,...,Ah,Dc,Dp,H,Igc,Igm,Sc,Ta,Ua,rho
I1,1.000,0.998,0.996,0.996,0.996,0.996,0.996,-0.967,-0.951,-0.791,...,0.092,0.040,0.117,0.067,-0.030,-0.028,0.036,0.040,0.090,-0.087
I2,0.998,1.000,0.995,0.996,0.996,0.996,0.994,-0.965,-0.951,-0.791,...,0.091,0.040,0.117,0.066,-0.030,-0.029,0.036,0.040,0.091,-0.086
I3,0.996,0.995,1.000,0.989,0.988,0.989,0.999,-0.972,-0.946,-0.788,...,0.094,0.038,0.119,0.070,-0.028,-0.026,0.034,0.043,0.088,-0.089
P,0.996,0.996,0.989,1.000,1.000,1.000,0.991,-0.948,-0.937,-0.763,...,0.092,0.039,0.117,0.069,-0.025,-0.023,0.035,0.044,0.084,-0.088
P1,0.996,0.996,0.988,1.000,1.000,1.000,0.990,-0.947,-0.937,-0.762,...,0.091,0.040,0.117,0.069,-0.025,-0.023,0.035,0.044,0.083,-0.088
P2,0.996,0.996,0.989,1.000,1.000,1.000,0.990,-0.947,-0.937,-0.763,...,0.091,0.040,0.116,0.068,-0.025,-0.024,0.036,0.043,0.084,-0.088
P3,0.996,0.994,0.999,0.991,0.990,0.990,1.000,-0.961,-0.933,-0.763,...,0.097,0.037,0.121,0.074,-0.022,-0.020,0.033,0.049,0.081,-0.093
PF,-0.967,-0.965,-0.972,-0.948,-0.947,-0.947,-0.961,1.000,0.985,0.900,...,-0.079,-0.042,-0.107,-0.045,0.058,0.056,-0.038,-0.013,-0.123,0.069
PF1,-0.951,-0.951,-0.946,-0.937,-0.937,-0.937,-0.933,0.985,1.000,0.929,...,-0.068,-0.048,-0.097,-0.030,0.064,0.062,-0.047,0.002,-0.136,0.058
PF2,-0.791,-0.791,-0.788,-0.763,-0.762,-0.763,-0.763,0.900,0.929,1.000,...,-0.030,-0.055,-0.060,0.024,0.137,0.136,-0.045,0.062,-0.190,0.008


## Example: Seasonal Full Matrix

In [7]:
corr_summer = show_meter_season('H1.Z16', 'Summer')
corr_summer.round(3)

meter_urn = H1.Z16 | season = Summer
rows = 13248
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


,I1,I2,I3,P,P1,P2,P3,PF,PF1,PF2,...,Ah,Dc,Dp,H,Igc,Igm,Sc,Ta,Ua,rho
I1,1.000,1.000,1.000,0.998,0.998,0.998,0.998,-0.952,-0.926,-0.790,...,0.072,0.006,0.073,0.072,0.022,0.023,0.032,0.034,0.031,-0.048
I2,1.000,1.000,1.000,0.998,0.998,0.998,0.998,-0.952,-0.926,-0.790,...,0.071,0.007,0.073,0.072,0.023,0.023,0.032,0.034,0.031,-0.048
I3,1.000,1.000,1.000,0.997,0.997,0.997,0.998,-0.954,-0.928,-0.793,...,0.072,0.007,0.073,0.072,0.023,0.023,0.032,0.034,0.031,-0.048
P,0.998,0.998,0.997,1.000,1.000,1.000,1.000,-0.931,-0.901,-0.751,...,0.069,0.004,0.070,0.075,0.026,0.027,0.032,0.040,0.025,-0.053
P1,0.998,0.998,0.997,1.000,1.000,1.000,1.000,-0.930,-0.900,-0.750,...,0.069,0.004,0.070,0.074,0.025,0.026,0.032,0.040,0.025,-0.053
P2,0.998,0.998,0.997,1.000,1.000,1.000,1.000,-0.930,-0.900,-0.750,...,0.068,0.005,0.070,0.074,0.026,0.027,0.033,0.040,0.025,-0.053
P3,0.998,0.998,0.998,1.000,1.000,1.000,1.000,-0.932,-0.902,-0.752,...,0.070,0.005,0.071,0.075,0.027,0.027,0.033,0.041,0.025,-0.054
PF,-0.952,-0.952,-0.954,-0.931,-0.930,-0.930,-0.932,1.000,0.996,0.936,...,-0.079,-0.015,-0.078,-0.054,-0.002,-0.003,-0.025,-0.001,-0.058,0.019
PF1,-0.926,-0.926,-0.928,-0.901,-0.900,-0.900,-0.902,0.996,1.000,0.953,...,-0.079,-0.021,-0.079,-0.053,-0.010,-0.010,-0.029,0.000,-0.058,0.017
PF2,-0.790,-0.790,-0.793,-0.751,-0.750,-0.750,-0.752,0.936,0.953,1.000,...,-0.078,-0.021,-0.074,-0.016,0.063,0.063,-0.001,0.052,-0.096,-0.031


## Batch Run Template

In [8]:
TARGET_METER = 'H1.Z16'
TARGET_YEAR = 2023
TARGET_SEASON = 'Summer'

show_meter_6year(TARGET_METER)
show_meter_year(TARGET_METER, TARGET_YEAR)
show_meter_season(TARGET_METER, TARGET_SEASON)

meter_urn = H1.Z16
measurement columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']
usable column count = 30
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


meter_urn = H1.Z16 | year = 2023
rows = 8760
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


meter_urn = H1.Z16 | season = Summer
rows = 13248
usable columns = ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'W_in', 'f', 'Ah', 'Dc', 'Dp', 'H', 'Igc', 'Igm', 'Sc', 'Ta', 'Ua', 'rho']


,I1,I2,I3,P,P1,P2,P3,PF,PF1,PF2,...,Ah,Dc,Dp,H,Igc,Igm,Sc,Ta,Ua,rho
I1,1.000000,0.999797,0.999832,0.997799,0.997666,0.997683,0.998016,-0.952097,-0.926266,-0.790250,...,0.071716,0.006496,0.072993,0.072098,0.022385,0.023143,0.031924,0.034043,0.030864,-0.048223
I2,0.999797,1.000000,0.999830,0.997796,0.997652,0.997695,0.997997,-0.952072,-0.926213,-0.790143,...,0.071438,0.006512,0.072728,0.072099,0.022687,0.023448,0.031968,0.034323,0.030553,-0.048413
I3,0.999832,0.999830,1.000000,0.997387,0.997225,0.997261,0.997823,-0.953664,-0.928079,-0.792830,...,0.071996,0.006704,0.073264,0.072159,0.022566,0.023313,0.031988,0.033859,0.031053,-0.048076
P,0.997799,0.997796,0.997387,1.000000,0.999981,0.999980,0.999758,-0.930840,-0.900866,-0.750861,...,0.068750,0.004456,0.070368,0.074598,0.025983,0.026828,0.032294,0.040438,0.024510,-0.053447
P1,0.997666,0.997652,0.997225,0.999981,1.000000,0.999982,0.999740,-0.930061,-0.899862,-0.749765,...,0.068583,0.004450,0.070206,0.074180,0.025327,0.026182,0.032382,0.040001,0.024866,-0.053174
P2,0.997683,0.997695,0.997261,0.999980,0.999982,1.000000,0.999734,-0.930245,-0.900106,-0.749874,...,0.068415,0.004566,0.070065,0.074260,0.025849,0.026701,0.032645,0.040281,0.024544,-0.053361
P3,0.998016,0.997997,0.997823,0.999758,0.999740,0.999734,1.000000,-0.931959,-0.901997,-0.752300,...,0.069661,0.004760,0.071330,0.075318,0.026528,0.027357,0.033014,0.040546,0.024826,-0.053877
PF,-0.952097,-0.952072,-0.953664,-0.930840,-0.930061,-0.930245,-0.931959,1.000000,0.996160,0.935690,...,-0.078778,-0.015086,-0.078135,-0.053824,-0.002250,-0.002605,-0.025190,-0.001454,-0.057920,0.019003
PF1,-0.926266,-0.926213,-0.928079,-0.900866,-0.899862,-0.900106,-0.901997,0.996160,1.000000,0.953012,...,-0.079457,-0.020695,-0.078917,-0.053245,-0.009742,-0.009865,-0.029466,0.000026,-0.058303,0.017437
PF2,-0.790250,-0.790143,-0.792830,-0.750861,-0.749765,-0.749874,-0.752300,0.935690,0.953012,1.000000,...,-0.078402,-0.020590,-0.074215,-0.015604,0.062869,0.062613,-0.001097,0.052301,-0.096174,-0.030849
